## 결정 트리 알고리즘을 계속 이용하면 test_set 에 맞게 모델이 적응할 수 있다. 

*근데 왜* -> score 과정에서는 모델이 바뀌지 않는 거 아닌가? 

--> 모델이 바뀌는 것이 아니다 -> 사람의 선택이 바뀐다. 

max_depth=1 로 학습 → test 점수 확인 → 0.80
max_depth=2 로 학습 → test 점수 확인 → 0.82
max_depth=3 로 학습 → test 점수 확인 → 0.84   ← "이게 제일 좋네!" 선택
max_depth=4 로 학습 → test 점수 확인 → 0.83
...


-> 즉 test_set 이 초기 설정을 결정하는 과정에 영향을 주는 것임. 




In [48]:
#검증과 test를 분류해서 test 는 정확히 한 번만 작동하도록 한다. 
import pandas as pd

import sklearn
sklearn.set_config(display = 'text')
wine = pd.read_csv('https://bit.ly/wine_csv_data')

In [49]:
data = wine[['alcohol', 'sugar', 'pH']]
target = wine['class']

In [50]:
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(data, target, test_size=0.2, random_state= 42)

sub_input, val_input, sub_target, val_target = train_test_split(train_input, train_target, test_size=0.2, random_state= 42)



In [51]:
print(sub_input.shape, val_input.shape)

(4157, 3) (1040, 3)


In [52]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target)
print(dt.score(sub_input, sub_target))
print(dt.score(val_input, val_target))


0.9971133028626413
0.864423076923077


검증 세트를 따로 만들 경우 데이터가 줄어 들어서 모델 향상에 쓸 데이터가 줄어드는 문제가 생김. 

교차 검증을 이용해서 안정적인 검증 점수를 얻고 훈련에도 많은 데이터를 사용할 수 있음. 


-> 훈련 세트를 나눠 가면서 검증을 한다. 

In [53]:
from sklearn.model_selection import cross_validate

scores = cross_validate(dt, train_input, train_target)
print(scores)

{'fit_time': array([0.02068758, 0.0155437 , 0.01882839, 0.01917601, 0.0157299 ]), 'score_time': array([0.0050478 , 0.00604939, 0.00602531, 0.00503039, 0.00424194]), 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}


In [54]:
import numpy as np 
print(np.mean(scores['test_score']))

0.855300214703487


In [55]:
##분할을 어떻게 할 것인지에 대해서는 정해진 것이 없음. 그래서 셔플을 하는 등의 작업을 통해서 분할 검증 set을 만든다. 

from sklearn.model_selection import StratifiedKFold
scores = cross_validate(dt, train_input, train_target, cv=StratifiedKFold())
print(np.mean(scores['test_score']))

0.855300214703487


In [56]:
##분할을 어떻게 할 것인지에 대해서는 정해진 것이 없음. 그래서 셔플을 하는 등의 작업을 통해서 분할 검증 set을 만든다. 

splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_validate(dt, train_input, train_target, cv=splitter)
print(np.mean(scores['test_score']))

0.8574181117533719


하이퍼 파라미터의 경우에는 사람이 직접 수정하면서 최적의 값을 구해야 한다.

-> 하나의 매개변수 값을 결정하고 다음 것으로 넘어가 결정을 하는 방식은 통하지 않는다. 서로 유기적으로 영향을 주고 받기 때문에 

-> ** 그리하여 나온 것이 그리드 서치이다. 그리드 서치를 통해서 다변수 상황에서 최적 값을 구할 수 있다.**


In [57]:
from sklearn.model_selection import GridSearchCV 
params = {'min_impurity_decrease': [0.0001, 0.0002, 0.0003,0.0004,0.0005]}

In [58]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)

In [ ]:
gs.fit(train_input, train_target)

In [ ]:
dt = gs.best_estimator_
print(dt.score(train_input, train_target))

0.9615162593804117


In [ ]:
print(gs.best_params_)

{'min_impurity_decrease': 0.0001}


In [ ]:
print(gs.cv_results_['mean_test_score'])

[0.86819297 0.86453617 0.86492226 0.86780891 0.86761605]


In [ ]:
params = {'min_impurity_decrease':np.arange(0.0001, 0.001, 0.0001),
            'max_depth' : range(5, 20, 1),
            'min_samples_split': range(2, 100,10)
        }
#numpy 의 arrange 함수는 첫 번째 매개변수 값에서 시작해서 두 번째 매개 변수 도달까지 세 번째 매개변수를 더해서 배열을 만든다. ->두 번째 매개 변수는 포함되지 않으므로 -개



In [ ]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs= -1)
gs.fit(train_input, train_target)

KeyboardInterrupt: 

In [ ]:
print(gs.best_params_)
print(max(gs.cv_results_['mean_test_score']))
